# MyDigitalTwin — 02_topology / 02 Shape of Me

Construit le graphe topologique cross-plateformes via TDA Mapper.

## Pipeline
```
Sources warehouse
  spotify_liked_songs  twitter_likes  tiktok_likes  tiktok_saves
  instagram_saved      instagram_likes
        ↓
Enrichissement (caches 01_enrich.ipynb)
  TikTok : og:description       Instagram : username + bio
        ↓
Imputation contextuelle (orphelins ±15 min)
        ↓
Embedding all-MiniLM-L6-v2 (384D, L2-normalisé)
        ↓
D_sem  = cosine distance (sur embeddings texte)
D_beh  = euclidean distance (sur features cycliques temporelles)
D_final = α·D_sem + (1-α)·D_beh     α = 0.5
        ↓
TDA Mapper × 3 lenses :
  lens_time      : timestamp normalisé  → évolution chronologique
  lens_platform  : OHE plateforme + PCA → ponts inter-plateformes
  lens_density   : KDE sur embedding 2D → zones d'attention
        ↓
warehouse/topology_{lens}_nodes.parquet
warehouse/topology_{lens}_edges.parquet
app/assets/topology_data.json
```

## Dépendances
```bash
pip install sentence-transformers umap-learn scikit-learn numpy pandas pyarrow
pip install git+https://github.com/lucasimi/tda-mapper-python.git
```

## Prérequis
- `01_enrich.ipynb` exécuté (caches TikTok + Instagram)
- `tiktok.ipynb` exécuté (tiktok_saves dans le warehouse)

In [1]:
import sys, os, json, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")

# ── Config paths ──────────────────────────────────────────────────────────────
_d = os.path.abspath('')
while not os.path.exists(os.path.join(_d, 'config.py')):
    _p = os.path.dirname(_d)
    if _p == _d: raise RuntimeError("config.py introuvable")
    _d = _p
sys.path.insert(0, _d)
from config import WAREHOUSE

APP_ASSETS = os.path.join(_d, "app", "assets")
os.makedirs(os.path.join(WAREHOUSE, "topology"), exist_ok=True)
os.makedirs(APP_ASSETS, exist_ok=True)

# ── Hyperparamètres ────────────────────────────────────────────────────────────
ALPHA          = 0.5   # poids sémantique (0.5 : équilibre sem/beh — TikTok sans texte)
N_INTERVALS    = 10    # Mapper : nb de découpes de l'espace filtre
OVERLAP_FRAC   = 0.5   # Mapper : chevauchement entre intervalles
DBSCAN_EPS     = 0.5   # DBSCAN : rayon de voisinage sur D_final [0,1]  ← 0.25 trop petit
DBSCAN_MIN_S   = 3     # DBSCAN : nb minimum de points par cluster       ← 5 trop restrictif
MAX_PER_SOURCE = 2000  # Cap par source pour éviter matrice ingérable en RAM
MAX_ITEMS_GRAPH = 20   # Items montrés par cluster dans le graphe compound

CHECKPOINT_DIR = os.path.join(_d, "data", "topology_checkpoint")

print(f"Hyperparamètres :")
print(f"  alpha={ALPHA}  n_intervals={N_INTERVALS}  overlap={OVERLAP_FRAC}")
print(f"  DBSCAN eps={DBSCAN_EPS}  min_samples={DBSCAN_MIN_S}")
print(f"  max_per_source={MAX_PER_SOURCE}  max_items_graph={MAX_ITEMS_GRAPH}")
print(f"  WAREHOUSE      = {WAREHOUSE}")
print(f"  CHECKPOINT_DIR = {CHECKPOINT_DIR}")

Hyperparamètres :
  alpha=0.5  n_intervals=10  overlap=0.5
  DBSCAN eps=0.5  min_samples=3
  max_per_source=2000  max_items_graph=20
  WAREHOUSE      = /opt/spark/data/warehouse
  CHECKPOINT_DIR = /opt/spark/data/topology_checkpoint


---
## 1. Chargement des sources

In [2]:
def load_table(name, cols, rename=None, timestamp_col=None):
    """Charge une table warehouse, retourne DataFrame ou None."""
    path = os.path.join(WAREHOUSE, name)
    if not os.path.exists(path):
        print(f"  ⚠️  {name} absent du warehouse")
        return None
    df = pd.read_parquet(path)
    df = df[[c for c in cols if c in df.columns]].copy()
    if rename:
        df = df.rename(columns=rename)
    if timestamp_col and timestamp_col in df.columns:
        df["timestamp_ms"] = pd.to_datetime(df[timestamp_col], errors="coerce").astype("int64") // 1_000_000
    print(f"  ✓ {name:<28} {len(df):>6,} lignes (brut)")
    return df


def cap_source(df, label, max_n=MAX_PER_SOURCE, sort_col="timestamp_ms"):
    """Plafonne une source à max_n items (les plus récents si timestamp dispo)."""
    if df is None or len(df) <= max_n:
        return df
    if sort_col in df.columns and df[sort_col].max() > 0:
        df = df.sort_values(sort_col, ascending=False).head(max_n)
    else:
        df = df.sample(max_n, random_state=42)
    print(f"    ↳ cap {label} → {len(df):,} items (max_per_source={max_n})")
    return df


print("Chargement des sources...")
dfs_all = []

# ── Spotify liked songs ───────────────────────────────────────────────────────
df = load_table("spotify_liked_songs",
                ["trackName", "artistName", "trackUri"],
                rename={"trackName": "_track", "artistName": "_artist", "trackUri": "url"})
if df is not None:
    df = cap_source(df, "spotify")
    df["text"]        = df["_track"] + " " + df["_artist"]
    df["label"]       = df["_track"] + " — " + df["_artist"]
    df["platform"]    = "spotify"
    df["action_type"] = "like"
    df["timestamp_ms"] = 0
    dfs_all.append(df[["text", "label", "url", "platform", "action_type", "timestamp_ms"]])

# ── Twitter likes ─────────────────────────────────────────────────────────────
df = load_table("twitter_likes",
                ["full_text", "post_url", "tweet_id"],
                rename={"full_text": "text", "post_url": "url", "tweet_id": "_tid"})
if df is not None:
    df["timestamp_ms"] = 0  # pas de timestamp dans twitter_likes
    df = cap_source(df, "twitter")     # ← cap : 140k → 2000
    df["label"]       = df["text"].str[:60] + "..."
    df["platform"]    = "twitter"
    df["action_type"] = "like"
    dfs_all.append(df[["text", "label", "url", "platform", "action_type", "timestamp_ms"]])

# ── TikTok likes ──────────────────────────────────────────────────────────────
df = load_table("tiktok_likes",
                ["video_id", "url", "timestamp_ms", "event_hour", "event_weekday"])
if df is not None:
    df = cap_source(df, "tiktok_likes")
    df["text"]        = None  # sera rempli depuis le cache
    df["label"]       = "TikTok #" + df["video_id"].str[:10]
    df["platform"]    = "tiktok"
    df["action_type"] = "like"
    df["_video_id"]   = df["video_id"]
    dfs_all.append(df[["text", "label", "url", "platform", "action_type",
                        "timestamp_ms", "event_hour", "event_weekday", "_video_id"]])

# ── TikTok saves ──────────────────────────────────────────────────────────────
df = load_table("tiktok_saves",
                ["video_id", "url", "timestamp_ms", "event_hour", "event_weekday"])
if df is not None:
    df = cap_source(df, "tiktok_saves")
    df["text"]        = None
    df["label"]       = "TikTok save #" + df["video_id"].str[:10]
    df["platform"]    = "tiktok"
    df["action_type"] = "save"
    df["_video_id"]   = df["video_id"]
    dfs_all.append(df[["text", "label", "url", "platform", "action_type",
                        "timestamp_ms", "event_hour", "event_weekday", "_video_id"]])

# ── Instagram saved ───────────────────────────────────────────────────────────
df = load_table("instagram_saved",
                ["account", "post_href", "timestamp", "event_hour", "event_weekday"],
                rename={"post_href": "url"})
if df is not None:
    df["timestamp_ms"] = pd.to_numeric(df["timestamp"], errors="coerce") * 1000
    df = cap_source(df, "instagram_saved")
    df["text"]         = None  # sera enrichi avec la bio
    df["label"]        = "@" + df["account"].fillna("")
    df["platform"]     = "instagram"
    df["action_type"]  = "save"
    df["_account"]     = df["account"]
    dfs_all.append(df[["text", "label", "url", "platform", "action_type",
                        "timestamp_ms", "event_hour", "event_weekday", "_account"]])

# ── Instagram likes ───────────────────────────────────────────────────────────
df = load_table("instagram_likes",
                ["post_url", "timestamp", "event_hour", "event_weekday"],
                rename={"post_url": "url"})
if df is not None:
    df["timestamp_ms"] = pd.to_numeric(df["timestamp"], errors="coerce") * 1000
    df = cap_source(df, "instagram_likes")  # ← cap : 53k → 2000
    df["text"]         = None  # pas de texte → comportemental
    df["label"]        = "Instagram like"
    df["platform"]     = "instagram"
    df["action_type"]  = "like"
    dfs_all.append(df[["text", "label", "url", "platform", "action_type",
                        "timestamp_ms", "event_hour", "event_weekday"]])

# ── Concat + index ────────────────────────────────────────────────────────────
df_all = pd.concat(dfs_all, ignore_index=True)
df_all["id"] = df_all.index.astype(str)

# Assurer les colonnes optionnelles
for col in ["event_hour", "event_weekday", "_video_id", "_account"]:
    if col not in df_all.columns:
        df_all[col] = None

df_all["event_hour"]    = pd.to_numeric(df_all["event_hour"],    errors="coerce").fillna(12)
df_all["event_weekday"] = pd.to_numeric(df_all["event_weekday"], errors="coerce").fillna(1)

print(f"\nDataFrame unifié : {len(df_all):,} items")
print(df_all.groupby("platform")["id"].count().to_string())
print(f"\n→ Matrice de distance estimée : {len(df_all)**2 * 4 / 1024**3:.1f} GB")

Chargement des sources...
  ✓ spotify_liked_songs             116 lignes (brut)
  ✓ twitter_likes                140,272 lignes (brut)
    ↳ cap twitter → 2,000 items (max_per_source=2000)
  ✓ tiktok_likes                 12,000 lignes (brut)
    ↳ cap tiktok_likes → 2,000 items (max_per_source=2000)
  ✓ tiktok_saves                     72 lignes (brut)
  ✓ instagram_saved                  34 lignes (brut)
  ✓ instagram_likes              53,446 lignes (brut)
    ↳ cap instagram_likes → 2,000 items (max_per_source=2000)

DataFrame unifié : 6,222 items
platform
instagram    2034
spotify       116
tiktok       2072
twitter      2000

→ Matrice de distance estimée : 0.1 GB


---
## 2. Enrichissement depuis les caches

In [3]:
# Charger les caches d'enrichissement
tiktok_cache = {}
instagram_cache = {}

tiktok_cache_path    = os.path.join(WAREHOUSE, "tiktok_descriptions.json")
instagram_cache_path = os.path.join(WAREHOUSE, "instagram_bios.json")

if os.path.isfile(tiktok_cache_path):
    with open(tiktok_cache_path, encoding="utf-8") as f:
        tiktok_cache = json.load(f)
    print(f"\u2713 TikTok cache    : {len(tiktok_cache):,} entr\u00e9es")
else:
    print("\u26a0\ufe0f  tiktok_descriptions.json absent \u2014 lancer 01_enrich.ipynb")

if os.path.isfile(instagram_cache_path):
    with open(instagram_cache_path, encoding="utf-8") as f:
        instagram_cache = json.load(f)
    print(f"\u2713 Instagram cache : {len(instagram_cache):,} entr\u00e9es")
else:
    print("\u26a0\ufe0f  instagram_bios.json absent \u2014 lancer 01_enrich.ipynb")

# Appliquer les enrichissements
# TikTok : remplir text depuis le cache video_id
mask_tiktok = df_all["_video_id"].notna()
df_all.loc[mask_tiktok, "text"] = df_all.loc[mask_tiktok, "_video_id"].map(
    lambda vid: tiktok_cache.get(str(vid)) if pd.notna(vid) else None
)

# Instagram saved : text = username + bio
mask_ig_saved = (df_all["platform"] == "instagram") & (df_all["action_type"] == "save")
df_all.loc[mask_ig_saved, "text"] = df_all.loc[mask_ig_saved, "_account"].map(
    lambda acc: instagram_cache.get(str(acc).lower().strip()) if pd.notna(acc) else None
)

n_with_text = df_all["text"].notna().sum()
n_orphan    = df_all["text"].isna().sum()
print(f"\nItems avec texte  : {n_with_text:,} ({100*n_with_text/len(df_all):.0f}%)")
print(f"Items orphelins   : {n_orphan:,} (fallback comportemental)")

✓ TikTok cache    : 6,065 entrées
✓ Instagram cache : 17 entrées

Items avec texte  : 2,150 (35%)
Items orphelins   : 4,072 (fallback comportemental)


---
## 3. Embedding + Imputation contextuelle des orphelins

**Checkpoint** : la première exécution (~5-10 min) sauvegarde `embeds_all.npy` + `df_all.parquet`
dans `data/topology_checkpoint/`. Les exécutions suivantes rechargent depuis ce checkpoint
(< 5 s) et sautent les sections 3a (embedding) et 3b (imputation).

Pour forcer un recalcul complet : supprimer le dossier `data/topology_checkpoint/`.

### 3a. Vérification du checkpoint

In [4]:
# ── Checkpoint : skip embedding si déjà calculé ───────────────────────────────
# Lance une seule fois (5-10 min) → sauvegarde dans data/topology_checkpoint/
# Relances suivantes : rechargement instantané (<5 s)
CHECKPOINT_LOADED = False

_emb_path = os.path.join(CHECKPOINT_DIR, "embeds_all.npy")
_df_path  = os.path.join(CHECKPOINT_DIR, "df_all.parquet")

if os.path.isfile(_emb_path) and os.path.isfile(_df_path):
    print("🔄 Checkpoint trouvé — chargement...")
    embeds_all = np.load(_emb_path)
    df_all     = pd.read_parquet(_df_path)
    N          = len(df_all)
    CHECKPOINT_LOADED = True
    print(f"✓ embeds_all : {embeds_all.shape}  ({embeds_all.nbytes / 1024**2:.0f} MB)")
    print(f"✓ df_all     : {N:,} items")
    print(f"  Plateformes : {df_all.groupby('platform')['id'].count().to_dict()}")
    print(f"\n→ Sections 3 (imputation) et 4a (embedding) ignorées ✓")
else:
    print("ℹ️  Pas de checkpoint — les sections 3 & 4a vont tourner (~5-10 min)")
    print(f"   Elles seront sauvegardées dans : {CHECKPOINT_DIR}")
    print(f"   Pour forcer un recalcul : supprimer {_emb_path}")

🔄 Checkpoint trouvé — chargement...
✓ embeds_all : (6222, 384)  (9 MB)
✓ df_all     : 6,222 items
  Plateformes : {'instagram': 2034, 'spotify': 116, 'tiktok': 2072, 'twitter': 2000}

→ Sections 3 (imputation) et 4a (embedding) ignorées ✓


In [5]:
if not CHECKPOINT_LOADED:
    from sentence_transformers import SentenceTransformer
    from sklearn.preprocessing import normalize

    print("Chargement du modèle all-MiniLM-L6-v2...")
    model = SentenceTransformer("all-MiniLM-L6-v2")
    print("✓ Modèle chargé")

    # ── Embed les items AVEC texte ─────────────────────────────────────────────
    idx_text   = df_all[df_all["text"].notna()].index
    texts      = df_all.loc[idx_text, "text"].tolist()

    print(f"Embedding de {len(texts):,} items avec texte...")
    embeds_text = model.encode(texts, batch_size=64, show_progress_bar=True)
    embeds_text = normalize(embeds_text)   # L2 normalisation obligatoire pour cosine
    print(f"✓ Embeddings : shape={embeds_text.shape}")

    # Préparer l'index position → idx (pour la matrice finale)
    idx_to_pos = {idx: i for i, idx in enumerate(idx_text)}

In [6]:
if not CHECKPOINT_LOADED:
    from sklearn.preprocessing import normalize

    WINDOW_MS = 15 * 60 * 1000  # ±15 minutes en millisecondes

    # Préparer un tableau d'embeddings complet (N × 384)
    _N    = len(df_all)
    _DIM  = embeds_text.shape[1]
    embeds_all = np.zeros((_N, _DIM), dtype=np.float32)

    # Remplir avec les embeddings calculés
    for i, idx in enumerate(idx_text):
        pos = df_all.index.get_loc(idx)
        embeds_all[pos] = embeds_text[i]

    # ── Imputation contextuelle pour les orphelins ────────────────────────────
    idx_orphan = df_all[df_all["text"].isna()].index
    print(f"Imputation de {len(idx_orphan):,} orphelins...")

    ts_arr   = df_all["timestamp_ms"].fillna(0).values
    hour_arr = df_all["event_hour"].values
    has_text  = df_all["text"].notna().values

    imputed_from_window  = 0
    imputed_from_hourly  = 0

    # Centroïdes horaires pré-calculés (fallback)
    hourly_centroids = {}
    for hour in range(24):
        hour_mask  = has_text & (hour_arr == hour)
        hour_idxs  = np.where(hour_mask)[0]
        if len(hour_idxs) > 0:
            hourly_centroids[hour] = embeds_all[hour_idxs].mean(axis=0)
    centroid_global = embeds_all[np.where(has_text)[0]].mean(axis=0)

    for idx in idx_orphan:
        pos = df_all.index.get_loc(idx)
        ts  = ts_arr[pos]
        h   = int(hour_arr[pos])

        if ts > 0:
            window_mask = has_text & (np.abs(ts_arr - ts) <= WINDOW_MS)
            window_idxs = np.where(window_mask)[0]
        else:
            window_idxs = np.array([], dtype=int)

        if len(window_idxs) > 0:
            if ts > 0:
                dists   = np.abs(ts_arr[window_idxs] - ts).astype(float) + 1
                weights = 1.0 / dists
            else:
                weights = np.ones(len(window_idxs))
            weights /= weights.sum()
            embeds_all[pos] = (embeds_all[window_idxs] * weights[:, None]).sum(axis=0)
            imputed_from_window += 1
        elif h in hourly_centroids:
            embeds_all[pos] = hourly_centroids[h]
            imputed_from_hourly += 1
        else:
            embeds_all[pos] = centroid_global
            imputed_from_hourly += 1

    # Re-normaliser tout
    embeds_all = normalize(embeds_all)

    print(f"✓ Imputation terminée")
    print(f"  → depuis fenêtre ±15 min  : {imputed_from_window:,}")
    print(f"  → depuis centroïde horaire : {imputed_from_hourly:,}")
    print(f"  Shape finale : {embeds_all.shape}")

    # ── Sauvegarder le checkpoint ──────────────────────────────────────────────
    os.makedirs(CHECKPOINT_DIR, exist_ok=True)
    np.save(os.path.join(CHECKPOINT_DIR, "embeds_all.npy"), embeds_all)
    df_all.to_parquet(os.path.join(CHECKPOINT_DIR, "df_all.parquet"), index=True)
    print(f"\n✓ Checkpoint sauvegardé → {CHECKPOINT_DIR}")
    print(f"   embeds_all.npy : {embeds_all.nbytes / 1024**2:.0f} MB")
    print(f"   df_all.parquet : {_N:,} items")
    print(f"   → La prochaine exécution rechargera depuis ce checkpoint")

# N toujours défini (qu'on ait chargé ou calculé)
N = len(df_all)
print(f"\nN = {N:,} items — matrice D : {N**2 * 4 / 1024**2:.0f} MB")


N = 6,222 items — matrice D : 148 MB


---
## 4. Matrices de distance hybride

$$D_{final}(i,j) = \alpha \cdot D_{sem}(i,j) + (1-\alpha) \cdot D_{beh}(i,j)$$

- $D_{sem}$ : distance cosinus sur les embeddings texte (sémantique)
- $D_{beh}$ : distance euclidienne sur les features temporelles cycliques
- $\alpha = 0.7$ par défaut (sémantique dominant)

In [7]:
from sklearn.metrics import pairwise_distances

print(f"Calcul D_sem ({N}×{N} cosine)...")
# Embeddings déjà L2-normalisés → cosine_dist = 1 - dot product
D_sem = (1.0 - embeds_all @ embeds_all.T).astype(np.float32)
D_sem = np.clip(D_sem, 0.0, 2.0)
print(f"✓ D_sem  min={D_sem.min():.3f}  max={D_sem.max():.3f}  mean={D_sem.mean():.3f}")

Calcul D_sem (6222×6222 cosine)...
✓ D_sem  min=0.000  max=1.214  mean=0.554


In [8]:
# ── D_beh : encodage cyclique temporel ────────────────────────────────────────
# sin/cos évite la discontinuité heure 23 → 0
hour_arr    = df_all["event_hour"].fillna(12).values.astype(float)
weekday_arr = df_all["event_weekday"].fillna(1).values.astype(float)

# Timestamp normalisé [0,1] pour capturer l'évolution long-terme
ts_raw  = df_all["timestamp_ms"].fillna(0).values.astype(float)
ts_min, ts_max = ts_raw.min(), ts_raw.max()
ts_norm = (ts_raw - ts_min) / (ts_max - ts_min + 1e-9)

beh_features = np.column_stack([
    np.sin(2 * np.pi * hour_arr    / 24),
    np.cos(2 * np.pi * hour_arr    / 24),
    np.sin(2 * np.pi * weekday_arr / 7),
    np.cos(2 * np.pi * weekday_arr / 7),
    ts_norm,
]).astype(np.float32)

print(f"Features comportementales : {beh_features.shape}")
print(f"Calcul D_beh ({N}×{N} euclidean)...")
D_beh = pairwise_distances(beh_features, metric="euclidean").astype(np.float32)
# Normaliser [0,1]
D_beh /= (D_beh.max() + 1e-9)
print(f"\u2713 D_beh  min={D_beh.min():.3f}  max={D_beh.max():.3f}  mean={D_beh.mean():.3f}")

Features comportementales : (6222, 5)
Calcul D_beh (6222×6222 euclidean)...
✓ D_beh  min=0.000  max=1.000  mean=0.579


In [9]:
# ── D_final ────────────────────────────────────────────────────────────────────
# Normaliser D_sem [0,1] avant la combinaison
D_sem_norm = D_sem / (D_sem.max() + 1e-9)

D_final = (ALPHA * D_sem_norm + (1 - ALPHA) * D_beh).astype(np.float32)
np.fill_diagonal(D_final, 0.0)  # diagonale strictement nulle

print(f"\u2713 D_final : shape={D_final.shape}")
print(f"  alpha={ALPHA}  → sem poids {ALPHA}  beh poids {1-ALPHA}")
print(f"  min={D_final.min():.3f}  max={D_final.max():.3f}  mean={D_final.mean():.3f}")
print(f"  RAM estimée : {D_final.nbytes / 1024**2:.0f} MB")

✓ D_final : shape=(6222, 6222)
  alpha=0.5  → sem poids 0.5  beh poids 0.5
  min=0.000  max=0.954  mean=0.518
  RAM estimée : 148 MB


---
## 5. TDA Mapper × 3 lenses

In [10]:
from tdamapper.core import MapperAlgorithm
from tdamapper.cover import CubicalCover
from sklearn.cluster import DBSCAN
from sklearn.decomposition import PCA
from sklearn.neighbors import KernelDensity
import umap

print("\u2713 TDA Mapper, UMAP, sklearn charg\u00e9s")

# ── Réduction UMAP 2D depuis D_final (pour KDE + visualisation optionnelle) ──
print("UMAP 2D depuis D_final (metric=precomputed)...")
reducer_2d = umap.UMAP(
    n_components=2,
    metric="precomputed",
    n_neighbors=min(15, N - 1),
    min_dist=0.1,
    random_state=42,
)
X_2d = reducer_2d.fit_transform(D_final)
print(f"\u2713 UMAP 2D : {X_2d.shape}")

✓ TDA Mapper, UMAP, sklearn chargés
UMAP 2D depuis D_final (metric=precomputed)...
✓ UMAP 2D : (6222, 2)


In [11]:
# ── Calcul des 3 lenses ────────────────────────────────────────────────────────

# Lens 1 : timestamp normalisé → évolution chronologique
lens_time = ts_norm.reshape(-1, 1)
print(f"lens_time     : shape={lens_time.shape}  [{lens_time.min():.3f}, {lens_time.max():.3f}]")

# Lens 2 : plateforme OHE → PCA 1D → ponts inter-plateformes
platform_ohe = pd.get_dummies(df_all["platform"]).values.astype(float)
pca_platform = PCA(n_components=1, random_state=42).fit_transform(platform_ohe)
lens_platform = (pca_platform - pca_platform.min()) / (pca_platform.max() - pca_platform.min() + 1e-9)
print(f"lens_platform : shape={lens_platform.shape}  [{lens_platform.min():.3f}, {lens_platform.max():.3f}]")

# Lens 3 : densité KDE sur UMAP 2D → zones d'attention concentrée
kde = KernelDensity(kernel="gaussian", bandwidth=0.5).fit(X_2d)
lens_density_raw = kde.score_samples(X_2d).reshape(-1, 1)
# Normaliser [0,1] (inverser pour que haute densité = valeur élevée)
lens_density = (lens_density_raw - lens_density_raw.min()) / (lens_density_raw.max() - lens_density_raw.min() + 1e-9)
print(f"lens_density  : shape={lens_density.shape}  [{lens_density.min():.3f}, {lens_density.max():.3f}]")

lens_time     : shape=(6222, 1)  [0.000, 1.000]
lens_platform : shape=(6222, 1)  [0.000, 1.000]
lens_density  : shape=(6222, 1)  [0.000, 1.000]


In [12]:
# ── Wrapper DBSCAN precomputed ─────────────────────────────────────────────────
# tda-mapper extrait D[bin_indices] rectangulaire → DBSCAN metric=precomputed échoue.
# Fix : on passe np.arange(N) comme "données". tda-mapper découpe ces indices
# → fit(X, y) reçoit les indices du bin → on extrait D[np.ix_(idx, idx)] carré.
#
# Structure réelle du graphe (diagnostic) :
#   - Attribut des nœuds : 'ids'  (pas 'data')
#   - Type : list de int  ex. [5, 18, 27, 51, 53]

class PrecomputedDBSCAN:
    """DBSCAN wrapper qui gère correctement les sous-matrices precomputed."""

    def __init__(self, D, eps=0.3, min_samples=2):
        self.D           = D
        self.eps         = eps
        self.min_samples = min_samples
        self.labels_     = None

    def get_params(self, deep=True):
        return {"D": self.D, "eps": self.eps, "min_samples": self.min_samples}

    def set_params(self, **params):
        for k, v in params.items():
            setattr(self, k, v)
        return self

    def fit(self, X, y=None):
        indices = np.asarray(X).ravel().astype(int)
        D_sub   = self.D[np.ix_(indices, indices)]
        db      = DBSCAN(eps=self.eps, min_samples=self.min_samples,
                         metric="precomputed")
        db.fit(D_sub)
        self.labels_ = db.labels_
        return self


# ── Fonction Mapper ────────────────────────────────────────────────────────────
def run_mapper(D, lens, n_intervals=N_INTERVALS, overlap=OVERLAP_FRAC,
               eps=DBSCAN_EPS, min_samples=DBSCAN_MIN_S):
    n          = len(D)
    idx_data   = np.arange(n, dtype=float).reshape(-1, 1)
    cover      = CubicalCover(n_intervals=n_intervals, overlap_frac=overlap)
    clustering = PrecomputedDBSCAN(D, eps=eps, min_samples=min_samples)
    mapper     = MapperAlgorithm(cover=cover, clustering=clustering)
    graph      = mapper.fit_transform(idx_data, lens.ravel())
    return graph


# ── graph_to_dfs ───────────────────────────────────────────────────────────────
def graph_to_dfs(graph, df_meta, lens_values, lens_name):
    """Convertit le graphe Mapper en deux DataFrames (nodes, edges).

    tda-mapper stocke les membres sous l'attribut 'ids' (list de int),
    pas 'data'. Vérifié par diagnostic sur le graphe réel.
    """
    platform_colors = {
        "spotify":   "#1DB954",
        "twitter":   "#1DA1F2",
        "tiktok":    "#69C9D0",
        "instagram": "#E1306C",
    }

    nodes_list = []
    for node_id, pt_ids in graph.nodes(data="ids"):
        if not pt_ids:
            continue

        sub      = df_meta.iloc[pt_ids]
        platform = sub["platform"].mode()[0] if not sub["platform"].empty else "unknown"

        group_lens  = lens_values.ravel()[pt_ids]
        median_lens = np.median(group_lens)
        central_idx = pt_ids[int(np.argmin(np.abs(group_lens - median_lens)))]
        label       = str(df_meta.iloc[central_idx].get("label", f"node_{node_id}"))[:80]
        items_info  = sub[["label", "url", "platform"]].head(10).to_dict("records")

        nodes_list.append({
            "id":        f"n{node_id}",
            "size":      len(pt_ids),
            "val":       max(3, min(50, 4 + 46 * len(pt_ids) // max(1, N // N_INTERVALS))),
            "label":     label,
            "platform":  platform,
            "color":     platform_colors.get(platform, "#888888"),
            "avg_lens":  float(np.mean(group_lens)),
            "lens_name": lens_name,
            "items":     items_info,
        })

    edges_list = []
    for u, v in graph.edges():
        u_ids  = set(graph.nodes[u].get("ids", []))
        v_ids  = set(graph.nodes[v].get("ids", []))
        shared = len(u_ids & v_ids)
        edges_list.append({"source": f"n{u}", "target": f"n{v}", "weight": shared})

    return pd.DataFrame(nodes_list), pd.DataFrame(edges_list)


print("✓ PrecomputedDBSCAN + run_mapper + graph_to_dfs définis")
print(f"  Attribut nœuds : 'ids'  (confirmé par diagnostic)")
print(f"  DBSCAN : metric=precomputed  eps={DBSCAN_EPS}  min_samples={DBSCAN_MIN_S}")
print(f"  Cover  : n_intervals={N_INTERVALS}  overlap_frac={OVERLAP_FRAC}")

✓ PrecomputedDBSCAN + run_mapper + graph_to_dfs définis
  Attribut nœuds : 'ids'  (confirmé par diagnostic)
  DBSCAN : metric=precomputed  eps=0.5  min_samples=3
  Cover  : n_intervals=10  overlap_frac=0.5


In [13]:
# ── Diagnostic : inspecter la structure réelle du graphe Mapper ────────────────
# Lance Mapper sur une seule lens (platform, la plus simple) et print tout
# ce que le graphe contient → permet de corriger graph_to_dfs en connaissance de cause.

import traceback as _tb

print("=== DIAGNOSTIC MAPPER ===\n")
try:
    _g = run_mapper(D_final, lens_platform)

    print(f"Type           : {type(_g)}")
    print(f"Nb nœuds       : {_g.number_of_nodes()}")
    print(f"Nb arêtes      : {_g.number_of_edges()}")
    print(f"Graph attrs    : {dict(_g.graph)}")

    # ── Structure des nœuds ──────────────────────────────────────────────────
    print(f"\n--- Nœuds (premiers 5) ---")
    for i, node in enumerate(_g.nodes()):
        attrs = dict(_g.nodes[node])
        print(f"  node_id={node!r}  type={type(node).__name__}")
        for k, v in attrs.items():
            sample = list(v)[:5] if hasattr(v, '__iter__') else v
            print(f"    attr '{k}' : type={type(v).__name__}  sample={sample}")
        if i >= 4:
            print("  ...")
            break

    if _g.number_of_nodes() == 0:
        print("  ⚠️  Aucun nœud dans le graphe — le clustering met tout en bruit (-1)")
        print("  Vérifions ce que DBSCAN retourne sur un bin :")
        # Simuler un bin manuellement
        _n  = len(D_final)
        _mid = _n // 2
        _sample_idx = np.arange(_mid, min(_mid + 200, _n))
        _D_sub = D_final[np.ix_(_sample_idx, _sample_idx)]
        from sklearn.cluster import DBSCAN as _DBSCAN
        _db = _DBSCAN(eps=DBSCAN_EPS, min_samples=DBSCAN_MIN_S, metric="precomputed")
        _db.fit(_D_sub)
        _lbls = _db.labels_
        print(f"  Sur 200 items (milieu) : labels uniques={np.unique(_lbls)}")
        print(f"  → noise (-1) : {(_lbls == -1).sum()}  clusters : {(_lbls >= 0).sum()}")
        print(f"  D_final sur ce bin : min={_D_sub.min():.3f}  max={_D_sub.max():.3f}  mean={_D_sub.mean():.3f}")
        print(f"\n  ➜ Si tout est -1, DBSCAN_EPS={DBSCAN_EPS} est trop petit pour cette échelle")
        print(f"    Valeur min de D_final : {D_final.min():.4f}  max : {D_final.max():.4f}  mean : {D_final.mean():.4f}")
        pct = np.percentile(D_final[D_final > 0], [5, 10, 25, 50])
        print(f"    Percentiles D_final>0 : 5%={pct[0]:.4f}  10%={pct[1]:.4f}  25%={pct[2]:.4f}  50%={pct[3]:.4f}")
        print(f"    → eps suggéré : entre {pct[1]:.3f} et {pct[2]:.3f}")

    # ── Structure des arêtes ──────────────────────────────────────────────────
    print(f"\n--- Arêtes (premières 5) ---")
    for i, (u, v, attrs) in enumerate(_g.edges(data=True)):
        print(f"  ({u!r}, {v!r})  attrs={attrs}")
        if i >= 4:
            break

except Exception as e:
    print(f"Erreur : {e}")
    _tb.print_exc()

=== DIAGNOSTIC MAPPER ===

Type           : <class 'networkx.classes.graph.Graph'>
Nb nœuds       : 4
Nb arêtes      : 0
Graph attrs    : {}

--- Nœuds (premiers 5) ---
  node_id=0  type=int64
    attr 'size' : type=int  sample=116
    attr 'ids' : type=list  sample=[0, 1, 2, 3, 4]
  node_id=1  type=int64
    attr 'size' : type=int  sample=2000
    attr 'ids' : type=list  sample=[116, 117, 118, 119, 120]
  node_id=2  type=int64
    attr 'size' : type=int  sample=2072
    attr 'ids' : type=list  sample=[2116, 2117, 2118, 2119, 2120]
  node_id=3  type=int64
    attr 'size' : type=int  sample=2034
    attr 'ids' : type=list  sample=[4188, 4189, 4190, 4191, 4192]

--- Arêtes (premières 5) ---


In [14]:
# ── Exécution des 3 lenses ─────────────────────────────────────────────────────
results = {}
lenses  = {
    "time":     lens_time,
    "platform": lens_platform,
    "density":  lens_density,
}

for lens_name, lens_val in lenses.items():
    print(f"\nMapper — lens={lens_name}...")
    try:
        g = run_mapper(D_final, lens_val)
        df_nodes, df_edges = graph_to_dfs(g, df_all, lens_val, lens_name)
        results[lens_name] = {"graph": g, "nodes": df_nodes, "edges": df_edges}
        n_nodes, n_edges = len(df_nodes), len(df_edges)
        print(f"  ✓ nœuds={n_nodes}  arêtes={n_edges}")
        if n_nodes == 0:
            print(f"  ⚠️  0 nœuds — DBSCAN_EPS trop petit ? Essayer eps={DBSCAN_EPS * 2:.2f}")
        elif n_nodes < 8:
            print(f"  ⚠️  Peu de nœuds — diminuer overlap ({OVERLAP_FRAC}→0.2) ou augmenter n_intervals ({N_INTERVALS}→12)")
        elif n_nodes > 80:
            print(f"  ⚠️  Trop de nœuds — augmenter overlap ({OVERLAP_FRAC}→0.4) ou diminuer n_intervals ({N_INTERVALS}→8)")
        else:
            print(f"  ✓ Calibration OK ({n_nodes} nœuds — cible 8-80)")
    except Exception as e:
        import traceback
        print(f"  ❌ Erreur : {e}")
        print(traceback.format_exc())
        results[lens_name] = {"nodes": pd.DataFrame(), "edges": pd.DataFrame()}


Mapper — lens=time...
  ✓ nœuds=2  arêtes=0
  ⚠️  Peu de nœuds — diminuer overlap (0.5→0.2) ou augmenter n_intervals (10→12)

Mapper — lens=platform...
  ✓ nœuds=4  arêtes=0
  ⚠️  Peu de nœuds — diminuer overlap (0.5→0.2) ou augmenter n_intervals (10→12)

Mapper — lens=density...
  ✓ nœuds=7  arêtes=3
  ⚠️  Peu de nœuds — diminuer overlap (0.5→0.2) ou augmenter n_intervals (10→12)


---
## 6. Export warehouse + JSON dashboard

In [15]:
topo_dir = os.path.join(WAREHOUSE, "topology")
os.makedirs(topo_dir, exist_ok=True)

for lens_name, res in results.items():
    df_nodes = res["nodes"]
    df_edges = res["edges"]

    if df_nodes.empty:
        print(f"\u26a0\ufe0f  lens={lens_name} vide, skip")
        continue

    # Parquet (sans la colonne 'items' qui contient des dicts)
    nodes_path = os.path.join(topo_dir, f"nodes_{lens_name}.parquet")
    edges_path = os.path.join(topo_dir, f"edges_{lens_name}.parquet")

    df_nodes.drop(columns=["items"], errors="ignore").to_parquet(nodes_path, index=False)
    df_edges.to_parquet(edges_path, index=False)

    print(f"\u2713 {lens_name:<10}  nodes={nodes_path}")
    print(f"{'':14}  edges={edges_path}")

print("\n\u2713 Warehouse topology mis à jour")

✓ time        nodes=/opt/spark/data/warehouse/topology/nodes_time.parquet
                edges=/opt/spark/data/warehouse/topology/edges_time.parquet
✓ platform    nodes=/opt/spark/data/warehouse/topology/nodes_platform.parquet
                edges=/opt/spark/data/warehouse/topology/edges_platform.parquet
✓ density     nodes=/opt/spark/data/warehouse/topology/nodes_density.parquet
                edges=/opt/spark/data/warehouse/topology/edges_density.parquet

✓ Warehouse topology mis à jour


In [16]:
# ── Dérivation image_url (Spotify API + YouTube _video_id) ───────────────────
# Appelé UNE fois avant l'export JSON. Rapide : 116 tracks Spotify + YouTube direct.
import re as _re, requests as _req
from dotenv import load_dotenv
load_dotenv(os.path.join(_d, '.env'))

def _derive_image_urls(df):
    df = df.copy()
    df['image_url'] = None

    # ── Spotify : album art via Web API (batch de 50) ─────────────────────────
    sp_mask = df['platform'] == 'spotify'
    if sp_mask.any():
        try:
            tok = _req.post(
                'https://accounts.spotify.com/api/token',
                auth=(os.getenv('SPOTIFY_CLIENT_ID'), os.getenv('SPOTIFY_CLIENT_SECRET')),
                data={'grant_type': 'client_credentials'}, timeout=10
            ).json()['access_token']
            hdrs = {'Authorization': f'Bearer {tok}'}

            uri_ids = df.loc[sp_mask, 'url'].str.extract(r'spotify:track:([A-Za-z0-9]+)')[0]
            valid   = uri_ids.dropna().unique().tolist()
            id2img  = {}
            for i in range(0, len(valid), 50):
                batch = ','.join(valid[i:i+50])
                tracks = _req.get(f'https://api.spotify.com/v1/tracks?ids={batch}',
                                  headers=hdrs, timeout=15).json().get('tracks') or []
                for t in tracks:
                    if t and t.get('album', {}).get('images'):
                        # Prendre la plus petite image (suffisant pour thumbnail)
                        id2img[t['id']] = t['album']['images'][-1]['url']

            def _simg(uri):
                m = _re.search(r'spotify:track:([A-Za-z0-9]+)', str(uri))
                return id2img.get(m.group(1)) if m else None

            df.loc[sp_mask, 'image_url'] = df.loc[sp_mask, 'url'].apply(_simg)
            print(f"✅ Spotify  : {df.loc[sp_mask,'image_url'].notna().sum()}/{sp_mask.sum()} images")
        except Exception as e:
            print(f"⚠️  Spotify images : {e}")

    # ── YouTube : thumbnail depuis _video_id (gratuit, sans API) ──────────────
    yt_mask = (df['platform'] == 'youtube') & df['_video_id'].notna() & \
              (df['_video_id'].astype(str).str.strip() != '') & \
              (df['_video_id'].astype(str).str.strip() != 'nan')
    if yt_mask.any():
        df.loc[yt_mask, 'image_url'] = df.loc[yt_mask, '_video_id'].apply(
            lambda v: f"https://img.youtube.com/vi/{v}/hqdefault.jpg"
        )
        print(f"✅ YouTube  : {yt_mask.sum()} thumbnails")

    n_total = df['image_url'].notna().sum()
    print(f"📸 Total image_url : {n_total} / {len(df)} ({100*n_total/len(df):.0f}%)")
    return df

df_all = _derive_image_urls(df_all)


⚠️  Spotify images : 'access_token'
📸 Total image_url : 0 / 6222 (0%)


In [17]:
# ── Export graphe compound (cluster nodes + item nodes) ───────────────────────
# Format JSON : deux types de nœuds dans le même graphe :
#   type=cluster  → gros nœud représentant le cluster TDA Mapper
#   type=item     → petit nœud = item individuel (avec thumbnail si dispo)
# Deux types d'arêtes :
#   type=cc  → cluster↔cluster (TDA Mapper edges)
#   type=ci  → cluster↔item    (appartenance)

compound_json = {}

for lens_name, res in results.items():
    graph    = res["graph"]
    n_tda    = graph.number_of_nodes()
    e_tda    = graph.number_of_edges()
    print(f"\n── Lens : {lens_name}  ({n_tda} clusters TDA, {e_tda} edges TDA)")

    nodes_out, links_out = [], []

    for node_id, pt_ids in graph.nodes(data='ids'):
        if not pt_ids:
            continue
        c_id    = f'c_{node_id}'
        ids_arr = np.array(list(pt_ids))
        sub     = df_all.iloc[ids_arr]
        dom_plt = sub['platform'].mode().iloc[0]

        # ── Cluster node ───────────────────────────────────────────────────
        nodes_out.append({
            'id':       c_id,
            'type':     'cluster',
            'label':    f'C{node_id} · {dom_plt}',
            'platform': dom_plt,
            'size':     int(len(ids_arr)),
            'val':      max(4, int(np.sqrt(len(ids_arr))) + 2),
        })

        # ── Item nodes : les MAX_ITEMS_GRAPH plus centraux du cluster ──────
        if len(ids_arr) > MAX_ITEMS_GRAPH:
            D_sub    = D_final[np.ix_(ids_arr, ids_arr)]
            central  = np.argsort(D_sub.mean(axis=1))[:MAX_ITEMS_GRAPH]
            show_idx = ids_arr[central]
        else:
            show_idx = ids_arr

        for gi in show_idx:
            row  = df_all.iloc[int(gi)]
            i_id = f'i_{row["id"]}'
            img  = row.get('image_url')
            # Nettoyer NaN float → None
            if isinstance(img, float) or img == 'nan' or img == 'None':
                img = None

            nodes_out.append({
                'id':        i_id,
                'type':      'item',
                'cluster':   c_id,
                'platform':  str(row.get('platform', '')),
                'text':      str(row.get('text', '') or '')[:120],
                'url':       str(row.get('url', '')),
                'image_url': str(img) if img else None,
                'ts':        str(row.get('timestamp_ms', '')),
            })
            links_out.append({
                'source': c_id,
                'target': i_id,
                'weight': 1,
                'type':   'ci',
            })

    # ── Arêtes cluster↔cluster depuis TDA Mapper ──────────────────────────
    for u, v in graph.edges():
        u_ids = set(graph.nodes[u].get('ids', []))
        v_ids = set(graph.nodes[v].get('ids', []))
        links_out.append({
            'source': f'c_{u}',
            'target': f'c_{v}',
            'weight': int(len(u_ids & v_ids)),
            'type':   'cc',
        })

    n_clusters   = sum(1 for n in nodes_out if n['type'] == 'cluster')
    n_item_nodes = sum(1 for n in nodes_out if n['type'] == 'item')
    n_cc_edges   = sum(1 for l in links_out if l['type'] == 'cc')

    compound_json[lens_name] = {
        'nodes': nodes_out,
        'links': links_out,
        'meta': {
            'n_items':      int(N),
            'n_clusters':   n_clusters,
            'n_item_nodes': n_item_nodes,
            'n_cc_edges':   n_cc_edges,
            'alpha':        ALPHA,
            'n_intervals':  N_INTERVALS,
            'overlap':      OVERLAP_FRAC,
            'eps':          DBSCAN_EPS,
        },
    }
    print(f"   → {n_clusters} clusters  {n_item_nodes} items affichés  {n_cc_edges} arêtes cc")

out_json = os.path.join(APP_ASSETS, "topology_data.json")
with open(out_json, "w", encoding="utf-8") as f:
    json.dump(compound_json, f, ensure_ascii=False, separators=(",", ":"), default=str)

print(f"\n✅ topology_data.json → {out_json}")
size_kb = os.path.getsize(out_json) // 1024
print(f"   Taille : {size_kb} KB")



── Lens : time  (2 clusters TDA, 0 edges TDA)
   → 2 clusters  40 items affichés  0 arêtes cc

── Lens : platform  (4 clusters TDA, 0 edges TDA)
   → 4 clusters  80 items affichés  0 arêtes cc

── Lens : density  (7 clusters TDA, 3 edges TDA)
   → 7 clusters  140 items affichés  3 arêtes cc

✅ topology_data.json → /opt/spark/app/assets/topology_data.json
   Taille : 66 KB


In [18]:
# ── Récapitulatif ─────────────────────────────────────────────────────────────
print("=" * 60)
print("  SHAPE OF ME — PIPELINE TERMINÉ")
print("=" * 60)
print(f"  Items total  : {N:,}")
print(f"  Alpha        : {ALPHA}  |  EPS={DBSCAN_EPS}  MIN_S={DBSCAN_MIN_S}")
print()
for k, v in compound_json.items():
    m = v["meta"]
    imgs = sum(1 for n in v["nodes"] if n.get("type") == "item" and n.get("image_url"))
    print(f"  [{k:<10}]  {m['n_clusters']:>3} clusters  "
          f"{m['n_item_nodes']:>4} items affichés  "
          f"{m['n_cc_edges']:>3} arêtes cc  "
          f"{imgs} thumbnails")
print()
print(f"  Outputs :")
print(f"    {os.path.join(WAREHOUSE, 'topology')}/")
print(f"    {out_json}")
print()
print("  → Ouvrir le dashboard → Shape of Me")
print()
# Avertissements calibration
for k, v in compound_json.items():
    if v["meta"]["n_cc_edges"] == 0:
        print(f"  ⚠️  [{k}] 0 arêtes cluster↔cluster — augmenter DBSCAN_EPS ou OVERLAP_FRAC")
    if v["meta"]["n_clusters"] < 5:
        print(f"  ⚠️  [{k}] peu de clusters ({v['meta']['n_clusters']}) — baisser DBSCAN_EPS")


  SHAPE OF ME — PIPELINE TERMINÉ
  Items total  : 6,222
  Alpha        : 0.5  |  EPS=0.5  MIN_S=3

  [time      ]    2 clusters    40 items affichés    0 arêtes cc  0 thumbnails
  [platform  ]    4 clusters    80 items affichés    0 arêtes cc  0 thumbnails
  [density   ]    7 clusters   140 items affichés    3 arêtes cc  0 thumbnails

  Outputs :
    /opt/spark/data/warehouse/topology/
    /opt/spark/app/assets/topology_data.json

  → Ouvrir le dashboard → Shape of Me

  ⚠️  [time] 0 arêtes cluster↔cluster — augmenter DBSCAN_EPS ou OVERLAP_FRAC
  ⚠️  [time] peu de clusters (2) — baisser DBSCAN_EPS
  ⚠️  [platform] 0 arêtes cluster↔cluster — augmenter DBSCAN_EPS ou OVERLAP_FRAC
  ⚠️  [platform] peu de clusters (4) — baisser DBSCAN_EPS


In [19]:
print(df_all.columns.tolist())
# + un exemple par plateforme
for p in df_all['platform'].unique():
    row = df_all[df_all['platform']==p].iloc[0]
    print(f"\n--- {p} ---")
    print(row[['url'] + [c for c in df_all.columns if 'image' in c.lower() or 'thumb' in c.lower() or 'cover' in c.lower() or 'media' in c.lower()]].to_dict())

['text', 'label', 'url', 'platform', 'action_type', 'timestamp_ms', 'event_hour', 'event_weekday', '_video_id', '_account', 'id', 'image_url']

--- spotify ---
{'url': 'spotify:track:62qPhTdER9X8P2gEpHXWzN', 'image_url': None}

--- twitter ---
{'url': 'https://twitter.com/i/web/status/2023443490219409697', 'image_url': None}

--- tiktok ---
{'url': 'https://www.tiktokv.com/share/video/7620909048401054998/', 'image_url': None}

--- instagram ---
{'url': 'https://www.instagram.com/p/DXZqLgGDC1E/', 'image_url': None}
